# Train YOLO26s

Small variant — primary recommendation. `BATCH=8`, `IMGSZ=1280`, `EPOCHS=150` (early stop @ epoch 118, patience=30).  
Outputs: `checkpoints/yolo26s/` · `results/metrics/yolo26s_metrics.csv` · `results/plots/yolo26s/`

| Split | mAP@50 | mAP@50-95 | Precision | Recall |
|---|---|---|---|---|
| Val  | 0.6462 | 0.3764 | 0.6981 | 0.5859 |
| Test | 0.5301 | 0.2876 | 0.5269 | 0.5928 |

| Class | Val AP@50 | Test AP@50 |
|---|---|---|
| Trichomonas vaginalis      | 0.5813 | 0.5100 |
| Bacterial vaginosis        | 0.7035 | 0.6838 |
| Candida spp.               | 0.3277 | 0.3730 |
| Actinomyces spp.           | 0.9721 | 0.5534 |

In [1]:
import os
import tempfile

os.makedirs('/dev/shm/t', exist_ok=True)
os.environ['TMPDIR'] = '/dev/shm/t'
tempfile.tempdir = '/dev/shm/t'


In [2]:
MODEL_VARIANT = 'yolo26s'
BATCH         = 8
EPOCHS        = 150
IMGSZ         = 1280
WORKERS       = 4
PATIENCE      = 30

from pathlib import Path
ROOT           = Path('../../../').resolve()
DATASET_YAML   = ROOT / 'models/organism_det/configs/dataset.yaml'
CHECKPOINT_DIR = ROOT / 'models/organism_det/checkpoints' / MODEL_VARIANT
METRICS_DIR    = ROOT / 'models/organism_det/results/metrics'
PLOTS_DIR      = ROOT / 'models/organism_det/results/plots' / MODEL_VARIANT

for d in [CHECKPOINT_DIR, METRICS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert DATASET_YAML.exists(), f'Run 00_data_converter.ipynb first. Missing: {DATASET_YAML}'

In [3]:
import subprocess, sys

for pkg in ['ultralytics', 'pandas', 'matplotlib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import ultralytics
print(f'ultralytics {ultralytics.__version__}')
print(f'torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB')

ultralytics 8.4.60
torch 2.12.0+cu130  |  CUDA: True
GPU: NVIDIA GeForce RTX 4090  |  VRAM: 23.6 GB


In [4]:
from ultralytics import YOLO

model = YOLO(f'{MODEL_VARIANT}.pt')

model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    project=str(CHECKPOINT_DIR.parent),
    name=MODEL_VARIANT,
    exist_ok=True,
    optimizer='auto',
    mosaic=1.0,
    close_mosaic=10,
    copy_paste=0.3,
    flipud=0.5,
    fliplr=0.5,
    degrees=15.0,
    hsv_h=0.05,
    hsv_s=0.3,
    hsv_v=0.1,
    scale=0.5,
    plots=True,
    save=True,
    save_period=50,
    verbose=True,
)

Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/dataset.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.05, hsv_s=0.3, hsv_v=0.1, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s, nbs=64, nms=Fa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ee95287a9e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

### Copy weights and training plots to results directories

In [5]:
import shutil

RUN_DIR = CHECKPOINT_DIR.parent / MODEL_VARIANT

for weight in ['best.pt', 'last.pt']:
    src = RUN_DIR / 'weights' / weight
    if src.exists():
        shutil.copy(src, CHECKPOINT_DIR / weight)

for fname in [
    'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'PR_curve.png', 'F1_curve.png', 'results.png', 'labels.jpg',
]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, PLOTS_DIR / fname)

print(f'Weights → {CHECKPOINT_DIR}')
print(f'Plots   → {PLOTS_DIR}')

Weights → /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/checkpoints/yolo26s
Plots   → /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/results/plots/yolo26s


### Evaluate on val and test sets

In [6]:
import pandas as pd
from ultralytics import YOLO as _YOLO

CLASS_NAMES = [
    'Trichomonas vaginalis',
    'Bacterial vaginosis flora shift',
    'Candida spp.',
    'Actinomyces spp.',
]

best_model = _YOLO(str(CHECKPOINT_DIR / 'best.pt'))

rows = []
for split in ['val', 'test']:
    r = best_model.val(data=str(DATASET_YAML), split=split, imgsz=IMGSZ, verbose=False)
    row = {
        'model':     MODEL_VARIANT,
        'split':     split,
        'mAP50':     round(float(r.box.map50), 4),
        'mAP50_95':  round(float(r.box.map),   4),
        'precision': round(float(r.box.mp),     4),
        'recall':    round(float(r.box.mr),     4),
    }
    for i, name in enumerate(CLASS_NAMES):
        key = f'AP50_{name.replace(" ", "_")}'
        row[key] = round(float(r.box.ap50[i]), 4) if i < len(r.box.ap50) else None
    rows.append(row)

metrics_df = pd.DataFrame(rows)
out_path   = METRICS_DIR / f'{MODEL_VARIANT}_metrics.csv'
metrics_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}\n')
print(metrics_df.to_string(index=False))

Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5089.4±2952.3 MB/s, size: 147.0 KB)
val: Scanning /home/pritamthapa/projects/cervical_cancer/ml-models/data/organisms_data/val/labels.cache... 47 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 13.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.3it/s 0.9s0.5s
                   all         47        143      0.698      0.586      0.646      0.376
Speed: 5.7ms preprocess, 8.2ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/pritamthapa/projects/cervical_cancer/ml-models/runs/detect/val-4
Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 350.1±55.5 MB

### Sample predictions on validation images

In [7]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CLASS_COLORS = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
val_images   = sorted((ROOT / 'data/organisms_data/val/images').glob('*.jpg'))
samples      = random.sample(val_images, min(8, len(val_images)))

fig, axes = plt.subplots(2, 4, figsize=(32, 14))
for ax, img_path in zip(axes.flat, samples):
    img   = Image.open(img_path)
    preds = best_model.predict(str(img_path), imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(img)
    for box in preds.boxes:
        cls  = int(box.cls)
        conf = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none'
        )
        ax.add_patch(rect)
        label = f'{CLASS_NAMES[cls][:14]} {conf:.2f}'
        ax.text(x1, y1 - 4, label, color=CLASS_COLORS[cls], fontsize=6, fontweight='bold')
    ax.axis('off')
    ax.set_title(img_path.stem, fontsize=8)

fig.suptitle(f'{MODEL_VARIANT.upper()} — Validation Sample Predictions', fontsize=14)
plt.tight_layout()
out_path = PLOTS_DIR / 'sample_predictions.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

<Figure size 3200x1400 with 8 Axes>

Saved → /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/results/plots/yolo26s/sample_predictions.png


### Results summary

In [8]:
val_row  = metrics_df[metrics_df.split == 'val'].iloc[0]
test_row = metrics_df[metrics_df.split == 'test'].iloc[0]

print(f'\n{"=" * 62}')
print(f'  {MODEL_VARIANT.upper()}  RESULTS')
print(f'{"=" * 62}')
print(f'  {"Metric":<24} {"Val":>8} {"Test":>8}')
print(f'  {"-" * 42}')
for metric, label in [("mAP50", "mAP@50"), ("mAP50_95", "mAP@50-95"),
                       ("precision", "Precision"), ("recall", "Recall")]:
    print(f'  {label:<24} {val_row[metric]:>8.4f} {test_row[metric]:>8.4f}')
print(f'\n  Per-class AP@50 (test):')
for name in CLASS_NAMES:
    key    = f'AP50_{name.replace(" ", "_")}'
    val    = test_row.get(key)
    suffix = '  <- critical' if 'Trichomonas' in name else ''
    v_str  = f'{val:.4f}' if val is not None else 'N/A'
    print(f'    {name:<42} {v_str}{suffix}')
print(f'{"=" * 62}')


  YOLO26S  RESULTS
  Metric                        Val     Test
  ------------------------------------------
  mAP@50                     0.6462   0.5301
  mAP@50-95                  0.3764   0.2876
  Precision                  0.6981   0.5269
  Recall                     0.5859   0.5928

  Per-class AP@50 (test):
    Trichomonas vaginalis                      0.5100  <- critical
    Bacterial vaginosis flora shift            0.6838
    Candida spp.                               0.3730
    Actinomyces spp.                           0.5534
